In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from spark_config import SparkConfig
from logger import LoggerConfig
from io_config import IOConfig
from platform_app import PlatformApp

In [2]:
logger = LoggerConfig().setup_logger(log_dir=IOConfig.LOG_DIR)
spark = SparkConfig.create_spark(app_name="Databricks Toturial", logger=logger, use_databricks=True)
app = PlatformApp(spark=spark, logger=logger, catalog_name="databricks_toturial")

2026-03-05 09:26:56 | INFO     | Logging configured: level=DEBUG, format=colored
2026-03-05 09:26:57 | INFO     | Connected to Databricks via Spark Connect.
2026-03-05 09:26:57 | INFO     | Initializing Data Platform...
2026-03-05 09:26:57 | INFO     | Spark session initialized


In [3]:
logger.info("=" * 80)
logger.info("[Stage 1/5] Initializing Unity Catalog and schemas...")
app.create_catalog(name_catalog="databricks_toturial")
# Create default volumes in bronze schema
for volume in app.DEFAULT_VOLUMES:
    app.create_volume(schema_name="bronze", volume_name=volume)
logger.info("[Stage 1/5] Catalog and schemas initialized successfully.")
logger.info("=" * 80)

2026-03-05 09:27:00 | INFO     | ================================================================================
2026-03-05 09:27:00 | INFO     | [Stage 1/5] Initializing Unity Catalog and schemas...
2026-03-05 09:27:00 | INFO     | Creating catalog: databricks_toturial
2026-03-05 09:27:02 | INFO     | CATALOG databricks_toturial created successfully.
2026-03-05 09:27:03 | INFO     | Creating schema: bronze
2026-03-05 09:27:05 | INFO     | SCHEMA bronze created successfully.
2026-03-05 09:27:05 | INFO     | Creating schema: silver
2026-03-05 09:27:06 | INFO     | SCHEMA silver created successfully.
2026-03-05 09:27:06 | INFO     | Creating schema: gold
2026-03-05 09:27:07 | INFO     | SCHEMA gold created successfully.
2026-03-05 09:27:07 | INFO     | Catalog setup completed.
2026-03-05 09:27:07 | INFO     | Creating volume: databricks_toturial.bronze.checkpoints
2026-03-05 09:27:08 | INFO     | Volume checkpoints created successfully.
2026-03-05 09:27:08 | INFO     | [Stage 1/5] Catal

In [4]:
# Source Data Upload
logger.info("=" * 80)
logger.info("[Stage 2/5] Uploading source datasets to staging volume...")
app.upload_local_to_uc_volume(local_base=IOConfig.DATASET_DIR, catalog="databricks_toturial",
                                schema="source", volume="source_data")
logger.info("[Stage 2/5] Source data upload completed.")
logger.info("=" * 80)

2026-03-05 09:27:09 | INFO     | ================================================================================
2026-03-05 09:27:09 | INFO     | [Stage 2/5] Uploading source datasets to staging volume...
2026-03-05 09:27:10 | INFO     | Creating schema: source
2026-03-05 09:27:11 | INFO     | Starting upload to Unity Catalog Volume...
2026-03-05 09:27:13 | INFO     | Skip (unchanged): /Volumes/databricks_toturial/source/source_data/movies/movies.csv
2026-03-05 09:27:13 | INFO     | Skip (unchanged): /Volumes/databricks_toturial/source/source_data/orders/orders.csv
2026-03-05 09:27:13 | INFO     | [Stage 2/5] Source data upload completed.
2026-03-05 09:27:13 | INFO     | ================================================================================


In [5]:
df = spark.read.csv(
    "/Volumes/databricks_toturial/source/source_data/movies/movies.csv",
    header=True,
    inferSchema=True
)

In [6]:
df.show(n = 10, truncate=False)

+-------------------------------------------+---------+------------+-----------+-------------------------+-------+--------+---------+--------+--------+
|title                                      |industry |release_year|imdb_rating|studio                   |budget |revenue |unit     |currency|language|
+-------------------------------------------+---------+------------+-----------+-------------------------+-------+--------+---------+--------+--------+
|Pather Panchali                            |Bollywood|1955        |8.3        |Government of West Bengal|70000.0|100000.0|Thousands|INR     |Bengali |
|Doctor Strange in the Multiverse of Madness|Hollywood|2022        |7          |Marvel Studios           |200.0  |954.8   |Millions |USD     |English |
|Thor: The Dark World                       |Hollywood|2013        |6.8        |Marvel Studios           |165.0  |644.8   |Millions |USD     |English |
|Thor: Ragnarok                             |Hollywood|2017        |7.9        |Marvel S

In [7]:
df.printSchema()

root
 |-- title: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- imdb_rating: string (nullable = true)
 |-- studio: string (nullable = true)
 |-- budget: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- unit: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- language: string (nullable = true)



In [8]:
df.count()

37

In [9]:
df.columns

['title',
 'industry',
 'release_year',
 'imdb_rating',
 'studio',
 'budget',
 'revenue',
 'unit',
 'currency',
 'language']

In [10]:
df.describe().show(truncate=False)

+-------+--------+---------+------------------+------------------+----------------+------------------+------------------+---------+--------+--------+
|summary|title   |industry |release_year      |imdb_rating       |studio          |budget            |revenue           |unit     |currency|language|
+-------+--------+---------+------------------+------------------+----------------+------------------+------------------+---------+--------+--------+
|count  |37      |37       |37                |37                |34              |37                |37                |37       |37      |37      |
|mean   |NULL    |NULL     |2007.027027027027 |7.919444444444445 |NULL            |2084.9751351351347|4117.135135135136 |NULL     |NULL    |NULL    |
|stddev |NULL    |NULL     |17.657995492263687|1.2049468143436146|NULL            |11477.487145324878|16372.462681608891|NULL     |NULL    |NULL    |
|min    |3 Idiots|Bollywood|1946              |1.9               |20th Century Fox|1.0              

In [11]:
df.summary().show(truncate=False)

+-------+--------+---------+------------------+------------------+----------------+------------------+------------------+---------+--------+--------+
|summary|title   |industry |release_year      |imdb_rating       |studio          |budget            |revenue           |unit     |currency|language|
+-------+--------+---------+------------------+------------------+----------------+------------------+------------------+---------+--------+--------+
|count  |37      |37       |37                |37                |34              |37                |37                |37       |37      |37      |
|mean   |NULL    |NULL     |2007.027027027027 |7.919444444444445 |NULL            |2084.9751351351347|4117.135135135136 |NULL     |NULL    |NULL    |
|stddev |NULL    |NULL     |17.657995492263687|1.2049468143436146|NULL            |11477.487145324878|16372.462681608891|NULL     |NULL    |NULL    |
|min    |3 Idiots|Bollywood|1946              |1.9               |20th Century Fox|1.0              

## Columns Filtering

In [13]:
df_trimed = df.select("title", "imdb_rating", "industry")
df_trimed.show(n=10,truncate=False)

+-------------------------------------------+-----------+---------+
|title                                      |imdb_rating|industry |
+-------------------------------------------+-----------+---------+
|Pather Panchali                            |8.3        |Bollywood|
|Doctor Strange in the Multiverse of Madness|7          |Hollywood|
|Thor: The Dark World                       |6.8        |Hollywood|
|Thor: Ragnarok                             |7.9        |Hollywood|
|Thor: Love and Thunder                     |6.8        |Hollywood|
|The Shawshank Redemption                   |9.3        |Hollywood|
|Interstellar                               |8.6        |Hollywood|
|The Pursuit of Happyness                   |8          |Hollywood|
|Gladiator                                  |8.5        |Hollywood|
|Titanic                                    |7.9        |Hollywood|
+-------------------------------------------+-----------+---------+
only showing top 10 rows


## Row Filtering

In [15]:
df_2000_2010 = df.filter((df["release_year"] >= 2000) & (df["release_year"] <= 2010))
df_2000_2010.show(n=10,truncate=False)

+------------------------+---------+------------+-----------+------------------------+------+-------+--------+--------+--------+
|title                   |industry |release_year|imdb_rating|studio                  |budget|revenue|unit    |currency|language|
+------------------------+---------+------------+-----------+------------------------+------+-------+--------+--------+--------+
|The Pursuit of Happyness|Hollywood|2006        |8          |Columbia Pictures       |55.0  |307.1  |Millions|USD     |English |
|Gladiator               |Hollywood|2000        |8.5        |Universal Pictures      |103.0 |460.5  |Millions|USD     |English |
|Avatar                  |Hollywood|2009        |7.8        |20th Century Fox        |237.0 |2847.0 |Millions|USD     |English |
|The Dark Knight         |Hollywood|2008        |9          |Syncopy                 |185.0 |1006.0 |Millions|USD     |English |
|3 Idiots                |Bollywood|2009        |8.4        |Vinod Chopra Films      |550.0 |4000

In [16]:
from pyspark.sql.functions import col
df_2000_2010 = df.filter((col("release_year") >= 2000) & (col("release_year") <= 2010))
df_2000_2010.show(n=10, truncate=False)

+------------------------+---------+------------+-----------+------------------------+------+-------+--------+--------+--------+
|title                   |industry |release_year|imdb_rating|studio                  |budget|revenue|unit    |currency|language|
+------------------------+---------+------------+-----------+------------------------+------+-------+--------+--------+--------+
|The Pursuit of Happyness|Hollywood|2006        |8          |Columbia Pictures       |55.0  |307.1  |Millions|USD     |English |
|Gladiator               |Hollywood|2000        |8.5        |Universal Pictures      |103.0 |460.5  |Millions|USD     |English |
|Avatar                  |Hollywood|2009        |7.8        |20th Century Fox        |237.0 |2847.0 |Millions|USD     |English |
|The Dark Knight         |Hollywood|2008        |9          |Syncopy                 |185.0 |1006.0 |Millions|USD     |English |
|3 Idiots                |Bollywood|2009        |8.4        |Vinod Chopra Films      |550.0 |4000

In [17]:
df.filter(col("studio") == "Marvel Studios").show(truncate=False)

+-------------------------------------------+---------+------------+-----------+--------------+------+-------+--------+--------+--------+
|title                                      |industry |release_year|imdb_rating|studio        |budget|revenue|unit    |currency|language|
+-------------------------------------------+---------+------------+-----------+--------------+------+-------+--------+--------+--------+
|Doctor Strange in the Multiverse of Madness|Hollywood|2022        |7          |Marvel Studios|200.0 |954.8  |Millions|USD     |English |
|Thor: The Dark World                       |Hollywood|2013        |6.8        |Marvel Studios|165.0 |644.8  |Millions|USD     |English |
|Thor: Ragnarok                             |Hollywood|2017        |7.9        |Marvel Studios|180.0 |854.0  |Millions|USD     |English |
|Thor: Love and Thunder                     |Hollywood|2022        |6.8        |Marvel Studios|250.0 |670.0  |Millions|USD     |English |
|Avengers: Endgame                

In [18]:
unique_industry = df.select("industry").distinct()
unique_industry.show(truncate=False)

+---------+
|industry |
+---------+
|Bollywood|
|Hollywood|
+---------+



In [19]:
from pyspark.sql.functions import countDistinct
df.select(countDistinct("industry")).show(truncate=False)

+------------------------+
|count(DISTINCT industry)|
+------------------------+
|2                       |
+------------------------+



In [20]:
from pyspark.sql.functions import *

df = df.withColumn("profit", round(col("revenue") - col("budget"), 2))
df.show(n=10, truncate=False)

+-------------------------------------------+---------+------------+-----------+-------------------------+-------+--------+---------+--------+--------+-------+
|title                                      |industry |release_year|imdb_rating|studio                   |budget |revenue |unit     |currency|language|profit |
+-------------------------------------------+---------+------------+-----------+-------------------------+-------+--------+---------+--------+--------+-------+
|Pather Panchali                            |Bollywood|1955        |8.3        |Government of West Bengal|70000.0|100000.0|Thousands|INR     |Bengali |30000.0|
|Doctor Strange in the Multiverse of Madness|Hollywood|2022        |7          |Marvel Studios           |200.0  |954.8   |Millions |USD     |English |754.8  |
|Thor: The Dark World                       |Hollywood|2013        |6.8        |Marvel Studios           |165.0  |644.8   |Millions |USD     |English |479.8  |
|Thor: Ragnarok                         

In [21]:
df = df.withColumnRenamed("revenue", "total_revenue")
df.printSchema()

root
 |-- title: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- imdb_rating: string (nullable = true)
 |-- studio: string (nullable = true)
 |-- budget: double (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- unit: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- language: string (nullable = true)
 |-- profit: double (nullable = true)

